# Agricultural Streaming Pipeline — Portfolio Demonstration

**Educational, synthetic data only.**

> **Execution warning:** This is an archival, sequential demonstration notebook, not a one-click clean-room deployment. Run cells in their intended stage on a fresh, dedicated Databricks Unity Catalog schema/volume. **Do not run all cells against already-populated Bronze/Silver/Quarantine tables.** Some early source-creation code overwrites `batch_001.csv`, and intermediate assertions require particular stage-specific counts. Reusing checkpoints with changed source files is unsafe. The `%restart_python` cell intentionally restarts the interpreter.

All execution outputs were cleared for safe publication; results are reported separately in README.


## Overview and Environment



# Agricultural Transaction Streaming Pipeline

## Project Objective

Develop an educational incremental data pipeline
using Apache Spark Structured Streaming and Delta Lake.

## Data Source

Synthetic agricultural financing transactions.

## Pipeline Capabilities

1. Ingest new transaction files.
2. Validate transaction schemas and values.
3. Separate valid and invalid records.
4. Write incremental results to Delta Lake.
5. Maintain streaming checkpoints.
6. Test restart and recovery behavior.
7. Monitor processing results.
8. Publish reproducible GitHub documentation.

## Tools

Python, PySpark, Spark Structured Streaming,
Delta Lake, Spark SQL, Databricks

## Data Integrity

All transaction data are synthetic.
No actual FCA institution records or customer
financial information are used.


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("PROJECT 3: INCREMENTAL STREAMING PIPELINE")
print("-----------------------------------------")

print("Spark version:", spark.version)

# Verify the catalog and schema
spark.sql("SHOW SCHEMAS IN workspace").show()

# Check available Unity Catalog volumes
spark.sql(
    "SHOW VOLUMES IN workspace.usda_training"
).show(truncate=False)

print("-----------------------------------------")
print("STREAMING ENVIRONMENT READY")


## Batch 001 — Synthetic Source and Initial Ingestion


In [ ]:

from pathlib import Path
import csv

# Project-specific directory inside the existing volume
base_path = Path(
    "/Volumes/workspace/usda_training/"
    "raw_files/agri_streaming_project"
)

input_path = base_path / "incoming"
input_path.mkdir(parents=True, exist_ok=True)

# Synthetic agricultural transactions
transactions = [
    ["TXN001", "2026-11-15", "Farm Equipment", 25000, "CA"],
    ["TXN002", "2026-11-15", "Crop Financing", 18000, "IA"],
    ["TXN003", "2026-11-15", "Livestock", 32000, "TX"],
    ["TXN004", "2026-11-15", "Farm Equipment", 12500, "NE"],
    ["TXN005", "2026-11-15", "Crop Financing", 22000, "KS"]
]

file_path = input_path / "batch_001.csv"

with file_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "transaction_id",
        "transaction_date",
        "loan_category",
        "amount",
        "state"
    ])
    writer.writerows(transactions)

print("Created:", file_path)
print("Transaction records:", len(transactions))
print("File exists:", file_path.exists())
print("SYNTHETIC TRANSACTION BATCH CREATED")


In [ ]:

from pyspark.sql import types as T

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType()),
    T.StructField("transaction_date", T.StringType()),
    T.StructField("loan_category", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("state", T.StringType())
])

batch_df = (
    spark.read
    .option("header", "true")
    .schema(transaction_schema)
    .csv(str(input_path))
)

print("Loaded records:", batch_df.count())
display(batch_df.orderBy("transaction_id"))

print("TRANSACTION DATA VALIDATION COMPLETED")


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql import types as T

base = (
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project"
)

input_path = f"{base}/incoming"
checkpoint_path = f"{base}/checkpoints/bronze_v1"

table_name = (
    "workspace.usda_training."
    "agri_streaming_bronze"
)

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType()),
    T.StructField("transaction_date", T.StringType()),
    T.StructField("loan_category", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("state", T.StringType())
])

# Read incoming CSV files as a stream
stream_df = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(transaction_schema)
    .load(input_path)
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

# Process available files and save to Delta
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()

print("STREAMING INGESTION COMPLETED")

# Verify the Bronze table
bronze_df = spark.table(table_name)

print("Bronze table rows:", bronze_df.count())

display(
    bronze_df.select(
        "transaction_id",
        "amount",
        "state",
        "source_file"
    )
)


## Batch 002 — Incremental Ingestion


In [ ]:

from pathlib import Path
import csv

input_path = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project/incoming"
)

new_transactions = [
    ["TXN006", "2026-11-16", "Crop Financing", 15000, "IA"],
    ["TXN007", "2026-11-16", "Farm Equipment", 28000, "CA"],
    ["TXN008", "2026-11-16", "Livestock", 19500, "TX"],
    ["TXN009", "2026-11-16", "Crop Financing", 31000, "KS"],
    ["TXN010", "2026-11-16", "Farm Equipment", 22500, "NE"]
]

second_file = input_path / "batch_002.csv"

# Avoid accidentally modifying an already processed file
if second_file.exists():
    print("Second batch already exists; not rewriting.")
else:
    with second_file.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "transaction_id",
            "transaction_date",
            "loan_category",
            "amount",
            "state"
        ])
        writer.writerows(new_transactions)

    print("Created:", second_file)

print("SECOND TRANSACTION BATCH READY")


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Existing project paths
base = (
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project"
)

input_path = f"{base}/incoming"
checkpoint_path = f"{base}/checkpoints/bronze_v1"

table_name = (
    "workspace.usda_training.agri_streaming_bronze"
)

# Recreate the transaction schema
transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType()),
    T.StructField("transaction_date", T.StringType()),
    T.StructField("loan_category", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("state", T.StringType())
])

# Recreate the streaming DataFrame
stream_df = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(transaction_schema)
    .load(input_path)
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

print("STREAMING DATAFRAME RECREATED")


In [ ]:

# Resume using the original checkpoint

query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()

print("SECOND STREAMING RUN COMPLETED")

# Verify the Bronze table
bronze_df = spark.table(table_name)

print("Total Bronze records:", bronze_df.count())

display(
    bronze_df.groupBy("source_file")
    .count()
    .orderBy("source_file")
)


## Initial Validation and Table Creation


In [ ]:

from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.usda_training.agri_streaming_bronze"
)

# Normalize transaction fields
validated_df = (
    bronze_df
    .withColumn(
        "transaction_date_parsed",
        F.to_date("transaction_date", "yyyy-MM-dd")
    )
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(
                F.col("transaction_id").isNull()
                | (F.trim(F.col("transaction_id")) == ""),
                "Missing transaction ID"
            ),
            F.when(
                F.col("transaction_date_parsed").isNull(),
                "Invalid transaction date"
            ),
            F.when(
                F.col("amount").isNull()
                | (F.col("amount") <= 0)
                | F.isnan(F.col("amount")),
                "Invalid transaction amount"
            ),
            F.when(
                ~F.col("loan_category").isin(
                    "Farm Equipment",
                    "Crop Financing",
                    "Livestock"
                )
                | F.col("loan_category").isNull(),
                "Invalid loan category"
            ),
            F.when(
                ~F.col("state").isin(
                    "CA", "IA", "TX", "NE", "KS"
                )
                | F.col("state").isNull(),
                "Invalid state"
            )
        )
    )
)

valid_df = validated_df.filter(
    F.col("validation_error") == ""
)

quarantine_df = validated_df.filter(
    F.col("validation_error") != ""
)

print("SILVER VALIDATION RESULTS")
print("-------------------------")
print("Bronze records:", bronze_df.count())
print("Valid records:", valid_df.count())
print("Quarantined records:", quarantine_df.count())

display(
    validated_df.select(
        "transaction_id",
        "amount",
        "state",
        "validation_error"
    ).orderBy("transaction_id")
)

print("SILVER VALIDATION COMPLETED")


In [ ]:

silver_table = (
    "workspace.usda_training.agri_streaming_silver"
)

if spark.catalog.tableExists(silver_table):
    print("Silver table already exists; no changes made.")
else:
    (
        valid_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
    print("Silver Delta table created.")

silver_count = spark.table(silver_table).count()

print("Silver records:", silver_count)
print("SILVER TABLE VERIFICATION COMPLETED")


In [ ]:

quarantine_table = (
    "workspace.usda_training.agri_streaming_quarantine"
)

if spark.catalog.tableExists(quarantine_table):
    print("Quarantine table already exists; no changes made.")
else:
    (
        quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(quarantine_table)
    )
    print("Quarantine Delta table created.")

quarantine_count = spark.table(
    quarantine_table
).count()

print("Quarantined records:", quarantine_count)
print("QUARANTINE TABLE VERIFICATION COMPLETED")


## Batch 003 — Invalid Records and Ingestion


In [ ]:

from pathlib import Path
import csv

input_dir = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project/incoming"
)

input_dir.mkdir(parents=True, exist_ok=True)

batch_003 = [
    ["TXN011", "2026-11-17", "Farm Equipment", 20000, "CA"],
    ["TXN012", "2026-11-17", "Crop Financing", -5000, "IA"],
    ["TXN013", "2026-11-17", "Livestock", 18000, "XX"],
    ["TXN014", "invalid-date", "Farm Equipment", 24000, "NE"],
    ["", "2026-11-17", "Crop Financing", 12000, "KS"]
]

file_path = input_dir / "batch_003.csv"

if file_path.exists():
    print("Batch 003 already exists; not rewriting.")
else:
    with file_path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "transaction_id",
            "transaction_date",
            "loan_category",
            "amount",
            "state"
        ])
        writer.writerows(batch_003)

    print("Created:", file_path)

print("INVALID TRANSACTION TEST BATCH READY")


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql import types as T

base = (
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project"
)

checkpoint = f"{base}/checkpoints/bronze_v1"

bronze_table = (
    "workspace.usda_training.agri_streaming_bronze"
)

schema = T.StructType([
    T.StructField("transaction_id", T.StringType()),
    T.StructField("transaction_date", T.StringType()),
    T.StructField("loan_category", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("state", T.StringType())
])

stream_df = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load(f"{base}/incoming")
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

if query.exception() is not None:
    raise RuntimeError(str(query.exception()))

print("STREAMING INGESTION COMPLETED")

bronze_df = spark.table(bronze_table)

print("Total Bronze records:", bronze_df.count())

display(
    bronze_df
    .withColumn(
        "batch_file",
        F.regexp_extract(
            "source_file",
            r"([^/]+\.csv)$",
            1
        )
    )
    .groupBy("batch_file")
    .count()
    .orderBy("batch_file")
)


## Validation and Idempotent Delta MERGE


In [ ]:

from pyspark.sql import functions as F

bronze_table = "workspace.usda_training.agri_streaming_bronze"
silver_table = "workspace.usda_training.agri_streaming_silver"
quarantine_table = "workspace.usda_training.agri_streaming_quarantine"

bronze_df = spark.table(bronze_table)

# Parse the date safely, including deliberately invalid dates
validated_df = (
    bronze_df
    .withColumn(
        "transaction_date_parsed",
        F.expr("try_to_date(transaction_date, 'yyyy-MM-dd')")
    )
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(
                F.col("transaction_id").isNull() |
                (F.trim("transaction_id") == ""),
                "Missing transaction ID"
            ),
            F.when(
                F.col("transaction_date_parsed").isNull(),
                "Invalid transaction date"
            ),
            F.when(
                F.col("amount").isNull() |
                F.isnan("amount") |
                (F.col("amount") <= 0),
                "Invalid transaction amount"
            ),
            F.when(
                F.col("loan_category").isNull() |
                (~F.col("loan_category").isin(
                    "Farm Equipment", "Crop Financing", "Livestock"
                )),
                "Invalid loan category"
            ),
            F.when(
                F.col("state").isNull() |
                (~F.col("state").isin(
                    "CA", "IA", "TX", "NE", "KS"
                )),
                "Invalid state"
            )
        )
    )
)

valid_df = validated_df.filter(
    F.col("validation_error") == ""
)

quarantine_df = validated_df.filter(
    F.col("validation_error") != ""
)

print("VALIDATION RESULTS")
print("------------------")
print("Bronze:", bronze_df.count())
print("Valid:", valid_df.count())
print("Invalid:", quarantine_df.count())

display(
    quarantine_df.select(
        "transaction_id",
        "transaction_date",
        "amount",
        "state",
        "validation_error"
    )
)

print("TRANSACTION VALIDATION COMPLETED")


In [ ]:

print("EXISTING DELTA TABLE COUNTS")
print("---------------------------")

print("Silver:", spark.table(silver_table).count())
print("Quarantine:", spark.table(quarantine_table).count())

print("TABLE COUNTS VERIFIED")


In [ ]:

from pyspark.sql import functions as F

print("VALID TRANSACTION DUPLICATE CHECK")
print("---------------------------------")

duplicate_ids = (
    valid_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_ids.count()

print("Duplicate valid transaction IDs:", duplicate_count)

assert duplicate_count == 0, (
    "Duplicate IDs found. Review before merging."
)

print("DUPLICATE CHECK PASSED")


In [ ]:

# Create a temporary view for valid transactions
valid_df.createOrReplaceTempView(
    "agri_valid_updates"
)

# Insert transactions not already present
spark.sql("""
MERGE INTO
    workspace.usda_training.agri_streaming_silver AS target
USING
    agri_valid_updates AS source
ON target.transaction_id = source.transaction_id
WHEN NOT MATCHED THEN
    INSERT *
""")

silver_count = spark.table(
    "workspace.usda_training.agri_streaming_silver"
).count()

print("Silver records after MERGE:", silver_count)
print("SILVER MERGE COMPLETED")


In [ ]:

quarantine_df.createOrReplaceTempView(
    "agri_quarantine_updates"
)

spark.sql("""
MERGE INTO
    workspace.usda_training.agri_streaming_quarantine AS target
USING
    agri_quarantine_updates AS source
ON
    target.source_file <=> source.source_file
    AND target.transaction_id <=> source.transaction_id
    AND target.transaction_date <=> source.transaction_date
    AND target.loan_category <=> source.loan_category
    AND target.amount <=> source.amount
    AND target.state <=> source.state
WHEN NOT MATCHED THEN
    INSERT *
""")

quarantine_count = spark.table(
    "workspace.usda_training.agri_streaming_quarantine"
).count()

print("Quarantined records after MERGE:", quarantine_count)
print("QUARANTINE MERGE COMPLETED")


In [ ]:

bronze_count = spark.table(
    "workspace.usda_training.agri_streaming_bronze"
).count()

silver_count = spark.table(
    "workspace.usda_training.agri_streaming_silver"
).count()

quarantine_count = spark.table(
    "workspace.usda_training.agri_streaming_quarantine"
).count()

print("FINAL TRANSACTION PROCESSING RESULTS")
print("------------------------------------")

print("Bronze:", bronze_count)
print("Silver:", silver_count)
print("Quarantine:", quarantine_count)

assert bronze_count == 15
assert silver_count == 11
assert quarantine_count == 4

print("DELTA MERGE VALIDATION PASSED")


In [ ]:

bronze_table = "workspace.usda_training.agri_streaming_bronze"
silver_table = "workspace.usda_training.agri_streaming_silver"
quarantine_table = "workspace.usda_training.agri_streaming_quarantine"

before_counts = {
    "Bronze": spark.table(bronze_table).count(),
    "Silver": spark.table(silver_table).count(),
    "Quarantine": spark.table(quarantine_table).count()
}

print("COUNTS BEFORE RETRY")
print(before_counts)

assert before_counts == {
    "Bronze": 15,
    "Silver": 11,
    "Quarantine": 4
}


In [ ]:

spark.sql("""
MERGE INTO workspace.usda_training.agri_streaming_silver AS target
USING agri_valid_updates AS source
ON target.transaction_id = source.transaction_id
WHEN NOT MATCHED THEN INSERT *
""")

print("SILVER MERGE RETRY COMPLETED")


In [ ]:

spark.sql("""
MERGE INTO workspace.usda_training.agri_streaming_quarantine AS target
USING agri_quarantine_updates AS source
ON
    target.source_file <=> source.source_file
    AND target.transaction_id <=> source.transaction_id
    AND target.transaction_date <=> source.transaction_date
    AND target.loan_category <=> source.loan_category
    AND target.amount <=> source.amount
    AND target.state <=> source.state
WHEN NOT MATCHED THEN INSERT *
""")

print("QUARANTINE MERGE RETRY COMPLETED")


In [ ]:

after_counts = {
    "Bronze": spark.table(bronze_table).count(),
    "Silver": spark.table(silver_table).count(),
    "Quarantine": spark.table(quarantine_table).count()
}

print("COUNTS AFTER RETRY")
print(after_counts)

assert after_counts == before_counts, (
    "Record counts changed after retry!"
)

silver_duplicates = (
    spark.table(silver_table)
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate Silver transaction IDs:", silver_duplicates)

assert silver_duplicates == 0

print("IDEMPOTENCY TEST PASSED")


## Batch 004 — Restart and Recovery


In [ ]:

from pathlib import Path
import csv

base = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project"
)

input_dir = base / "incoming"
batch_file = input_dir / "batch_004.csv"

new_transactions = [
    ["TXN016", "2026-11-18", "Farm Equipment", 17500, "CA"],
    ["TXN017", "2026-11-18", "Crop Financing", 21000, "IA"],
    ["TXN018", "2026-11-18", "Livestock", 26500, "TX"]
]

if batch_file.exists():
    print("Batch 004 already exists; not rewriting.")
else:
    with batch_file.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "transaction_id",
            "transaction_date",
            "loan_category",
            "amount",
            "state"
        ])
        writer.writerows(new_transactions)

    print("Created:", batch_file)

print("BATCH 004 READY")


In [ ]:

%restart_python


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql import types as T

base = (
    "/Volumes/workspace/usda_training/raw_files/"
    "agri_streaming_project"
)

bronze_table = (
    "workspace.usda_training.agri_streaming_bronze"
)

checkpoint = f"{base}/checkpoints/bronze_v1"

schema = T.StructType([
    T.StructField("transaction_id", T.StringType()),
    T.StructField("transaction_date", T.StringType()),
    T.StructField("loan_category", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("state", T.StringType())
])

# Recreate the streaming source after restart
stream_df = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load(f"{base}/incoming")
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

# Resume using the ORIGINAL checkpoint
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

if query.exception() is not None:
    raise RuntimeError(str(query.exception()))

print("STREAMING RECOVERY COMPLETED")


In [ ]:

from pyspark.sql import functions as F

bronze_table = (
    "workspace.usda_training.agri_streaming_bronze"
)

bronze_df = spark.table(bronze_table)

total = bronze_df.count()

batch_counts = (
    bronze_df
    .withColumn(
        "batch_file",
        F.regexp_extract(
            "source_file",
            r"([^/]+\.csv)$",
            1
        )
    )
    .groupBy("batch_file")
    .count()
    .orderBy("batch_file")
)

print("STREAMING RECOVERY VALIDATION")
print("-----------------------------")
print("Total Bronze records:", total)

batch_counts.show(truncate=False)

assert total == 18, (
    f"Expected 18 records, found {total}"
)

expected = {
    "batch_001.csv": 5,
    "batch_002.csv": 5,
    "batch_003.csv": 5,
    "batch_004.csv": 3
}

actual = {
    row["batch_file"]: row["count"]
    for row in batch_counts.collect()
}

assert actual == expected, (
    f"Unexpected batch counts: {actual}"
)

print("CHECKPOINT RECOVERY TEST PASSED")


In [ ]:

from pyspark.sql import functions as F

bronze_table = (
    "workspace.usda_training.agri_streaming_bronze"
)

silver_table = (
    "workspace.usda_training.agri_streaming_silver"
)

quarantine_table = (
    "workspace.usda_training.agri_streaming_quarantine"
)

# Select only records from Batch 004
batch004 = (
    spark.table(bronze_table)
    .filter(
        F.col("source_file").endswith(
            "/batch_004.csv"
        )
    )
)

# Validate the new batch
validated004 = (
    batch004
    .withColumn(
        "transaction_date_parsed",
        F.expr(
            "try_cast(transaction_date AS DATE)"
        )
    )
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(
                F.col("transaction_id").isNull() |
                (F.trim("transaction_id") == ""),
                "Missing transaction ID"
            ),
            F.when(
                F.col("transaction_date_parsed").isNull(),
                "Invalid transaction date"
            ),
            F.when(
                F.col("amount").isNull() |
                F.isnan("amount") |
                (F.col("amount") <= 0),
                "Invalid transaction amount"
            ),
            F.when(
                F.col("loan_category").isNull() |
                (~F.col("loan_category").isin(
                    "Farm Equipment",
                    "Crop Financing",
                    "Livestock"
                )),
                "Invalid loan category"
            ),
            F.when(
                F.col("state").isNull() |
                (~F.col("state").isin(
                    "CA", "IA", "TX", "NE", "KS"
                )),
                "Invalid state"
            )
        )
    )
)

valid004 = validated004.filter(
    F.col("validation_error") == ""
)

invalid004 = validated004.filter(
    F.col("validation_error") != ""
)

print("BATCH 004 VALIDATION")
print("--------------------")
print("Total records:", batch004.count())
print("Valid records:", valid004.count())
print("Invalid records:", invalid004.count())

assert batch004.count() == 3
assert valid004.count() == 3
assert invalid004.count() == 0

print("BATCH 004 VALIDATION PASSED")


In [ ]:

# Check for duplicate IDs within this batch
duplicate_ids = (
    valid004
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
    .count()
)

assert duplicate_ids == 0, (
    "Duplicate transaction IDs found"
)

# Create a temporary view for Delta MERGE
valid004.createOrReplaceTempView(
    "agri_batch004_valid"
)

# Insert only transactions not already in Silver
spark.sql("""
MERGE INTO
    workspace.usda_training.agri_streaming_silver AS target
USING
    agri_batch004_valid AS source
ON target.transaction_id = source.transaction_id
WHEN NOT MATCHED THEN INSERT *
""")

print("BATCH 004 SILVER MERGE COMPLETED")

print("Bronze:", spark.table(bronze_table).count())
print("Silver:", spark.table(silver_table).count())
print("Quarantine:", spark.table(quarantine_table).count())

assert spark.table(bronze_table).count() == 18
assert spark.table(silver_table).count() == 14
assert spark.table(quarantine_table).count() == 4

print("INCREMENTAL SILVER UPDATE PASSED")


## Audit and Monitoring View


In [ ]:

from pyspark.sql import functions as F

bronze = spark.table(
    "workspace.usda_training.agri_streaming_bronze"
)

silver = spark.table(
    "workspace.usda_training.agri_streaming_silver"
)

quarantine = spark.table(
    "workspace.usda_training.agri_streaming_quarantine"
)

# Extract the filename associated with each record
def add_batch_name(df):
    return df.withColumn(
        "batch_file",
        F.regexp_extract(
            F.col("source_file"),
            r"([^/]+\.csv)$",
            1
        )
    )

# Count records by batch and processing layer
bronze_counts = (
    add_batch_name(bronze)
    .groupBy("batch_file")
    .agg(F.count("*").alias("bronze_records"))
)

silver_counts = (
    add_batch_name(silver)
    .groupBy("batch_file")
    .agg(F.count("*").alias("silver_records"))
)

quarantine_counts = (
    add_batch_name(quarantine)
    .groupBy("batch_file")
    .agg(F.count("*").alias("quarantine_records"))
)

# Combine all three layers
audit_df = (
    bronze_counts
    .join(silver_counts, "batch_file", "left")
    .join(quarantine_counts, "batch_file", "left")
    .fillna(0)
    .withColumn(
        "unaccounted_records",
        F.col("bronze_records")
        - F.col("silver_records")
        - F.col("quarantine_records")
    )
    .orderBy("batch_file")
)

print("AGRICULTURAL STREAMING PIPELINE AUDIT")
print("-------------------------------------")

display(audit_df)

# Verify reconciliation across all batches
totals = audit_df.agg(
    F.sum("bronze_records").alias("bronze"),
    F.sum("silver_records").alias("silver"),
    F.sum("quarantine_records").alias("quarantine"),
    F.sum("unaccounted_records").alias("unaccounted")
).first()

print("Bronze total:", totals["bronze"])
print("Silver total:", totals["silver"])
print("Quarantine total:", totals["quarantine"])
print("Unaccounted records:", totals["unaccounted"])

assert totals["bronze"] == 18
assert totals["silver"] == 14
assert totals["quarantine"] == 4
assert totals["unaccounted"] == 0

print("PIPELINE AUDIT VALIDATION PASSED")


In [ ]:

view_name = (
    "workspace.usda_training."
    "agri_streaming_batch_audit"
)

spark.sql(f"""
CREATE OR REPLACE VIEW {view_name} AS
WITH bronze_counts AS (
    SELECT
        regexp_extract(source_file, '([^/]+[.]csv)$', 1)
            AS batch_file,
        COUNT(*) AS bronze_records
    FROM workspace.usda_training.agri_streaming_bronze
    GROUP BY 1
),
silver_counts AS (
    SELECT
        regexp_extract(source_file, '([^/]+[.]csv)$', 1)
            AS batch_file,
        COUNT(*) AS silver_records
    FROM workspace.usda_training.agri_streaming_silver
    GROUP BY 1
),
quarantine_counts AS (
    SELECT
        regexp_extract(source_file, '([^/]+[.]csv)$', 1)
            AS batch_file,
        COUNT(*) AS quarantine_records
    FROM workspace.usda_training.agri_streaming_quarantine
    GROUP BY 1
)
SELECT
    b.batch_file,
    b.bronze_records,
    COALESCE(s.silver_records, 0) AS silver_records,
    COALESCE(q.quarantine_records, 0)
        AS quarantine_records,
    b.bronze_records
        - COALESCE(s.silver_records, 0)
        - COALESCE(q.quarantine_records, 0)
        AS unaccounted_records
FROM bronze_counts b
LEFT JOIN silver_counts s
    ON b.batch_file = s.batch_file
LEFT JOIN quarantine_counts q
    ON b.batch_file = q.batch_file
""")

print("PERSISTENT MONITORING VIEW CREATED")
print(view_name)


In [ ]:

from pyspark.sql import functions as F

view_name = (
    "workspace.usda_training."
    "agri_streaming_batch_audit"
)

audit = spark.table(view_name)

display(audit.orderBy("batch_file"))

totals = audit.agg(
    F.sum("bronze_records").alias("bronze"),
    F.sum("silver_records").alias("silver"),
    F.sum("quarantine_records").alias("quarantine"),
    F.sum("unaccounted_records").alias("unaccounted")
).first()

print("MONITORING VIEW VALIDATION")
print("--------------------------")
print("Bronze:", totals["bronze"])
print("Silver:", totals["silver"])
print("Quarantine:", totals["quarantine"])
print("Unaccounted:", totals["unaccounted"])

assert totals["bronze"] == 18
assert totals["silver"] == 14
assert totals["quarantine"] == 4
assert totals["unaccounted"] == 0

print("MONITORING VIEW VALIDATION PASSED")


## Final QA Results


In [ ]:

from pyspark.sql import functions as F

silver = spark.table(
    "workspace.usda_training.agri_streaming_silver"
)

quarantine = spark.table(
    "workspace.usda_training.agri_streaming_quarantine"
)

audit = spark.table(
    "workspace.usda_training.agri_streaming_batch_audit"
)

# Check Silver IDs in a single aggregation
silver_quality = silver.agg(
    F.count("*").alias("total"),
    F.countDistinct("transaction_id").alias("unique_ids"),
    F.sum(
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim("transaction_id") == ""),
            1
        ).otherwise(0)
    ).alias("missing_ids")
).first()

# Check quarantine explanations
quarantine_quality = quarantine.agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            F.col("validation_error").isNull() |
            (F.trim("validation_error") == ""),
            1
        ).otherwise(0)
    ).alias("missing_reasons")
).first()

# Check batch reconciliation
audit_quality = audit.agg(
    F.sum("bronze_records").alias("bronze"),
    F.sum("silver_records").alias("silver"),
    F.sum("quarantine_records").alias("quarantine"),
    F.sum("unaccounted_records").alias("unaccounted")
).first()

checks = {
    "Bronze contains 18 records":
        audit_quality["bronze"] == 18,

    "Silver contains 14 records":
        silver_quality["total"] == 14,

    "Quarantine contains 4 records":
        quarantine_quality["total"] == 4,

    "No duplicate Silver transaction IDs":
        silver_quality["unique_ids"] == 14,

    "No missing Silver transaction IDs":
        silver_quality["missing_ids"] == 0,

    "All quarantined records have reasons":
        quarantine_quality["missing_reasons"] == 0,

    "All batch records reconcile":
        audit_quality["unaccounted"] == 0
}

print("PROJECT 3 FINAL QUALITY ASSURANCE")
print("---------------------------------")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

assert all(checks.values()), "Quality assurance failed"

print("---------------------------------")
print("ALL PROJECT 3 QUALITY CHECKS PASSED")
